# 01: EDA and data audit (HKUST primary data)

This notebook is built up task by task (Stage 1 in `PLAN.md`). Reusable code is in `src/`; the cells here stay short and simple.

## 1. Setup

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src import inventory

## 2. Station inventory

One row per site-level power file. File facts (resolution, dates, rows) come from the CSVs. Rated power, tilt, azimuth and location come from the `.ttl` metadata, matched by station name (all 60 match; one name, *Indoor Sports Centre*, needed a manual fix).

In [2]:
inv = inventory.build_inventory()
inv.shape

(60, 18)

In [3]:
inv.head()

,station,has_optimizer,resolution_min,start,end,n_rows,n_duplicate_timestamps,n_inverter_files,ttl_name,latitude,longitude,altitude_m,connection_date,rated_unit,rated_value,azimuth_raw,tilt_raw,contractor
0,Indoor Sports Centre,True,15.0,2021-12-02,2023-12-31 23:45:00,72960,0,1,Indoor_Sport_Center,22.28,114.17,3.0,2021-12-01T00:00:00,KW,33.3,0/180deg,10deg,SolarEdge
1,LSK North,True,15.0,2021-06-01,2023-12-31 23:45:00,90624,0,1,LSK_North,22.33,114.18,142.0,2021-04-30T00:00:00,KW,55.0,10/-153deg,10deg,SolarEdge
2,LSK South,True,15.0,2021-06-01,2023-12-31 23:45:00,90624,0,1,LSK_South,22.33,114.18,152.0,2021-04-30T00:00:00,KW,25.0,0deg,0deg,SolarEdge
3,Library,True,15.0,2022-06-01,2023-12-31 23:45:00,55584,0,3,Library,22.28,114.17,87.0,2022-05-31 00:00:00,KW,170.2,Mixed,10deg,SolarEdge
4,S H Ho Sports Hall,True,15.0,2021-11-01,2023-12-31 23:45:00,75936,0,2,S_H_Ho_Sports_Hall,22.28,114.17,96.0,2021-10-02 00:00:00,KW,42.0,0deg,0deg,SolarEdge


### Checks on the join

In [4]:
print('files without .ttl match:', inv['ttl_name'].isna().sum())
print('rated power units:', inv['rated_unit'].unique())
print('inverter files:', inv['n_inverter_files'].sum())

files without .ttl match: 0
rated power units: <StringArray>
['KW']
Length: 1, dtype: str
inverter files: 44


In [5]:
inv.groupby('has_optimizer')[['resolution_min']].agg(['count', 'min', 'max'])

resolution_min            
                       count   min   max
has_optimizer                           
False                     23  60.0  60.0
True                      37  15.0  15.0

### Stations with problems to note

In [6]:
inv[inv['n_duplicate_timestamps'] > 0][['station', 'n_rows', 'n_duplicate_timestamps']]

,station,n_rows,n_duplicate_timestamps
39,SQ Apartment37-48 Flexible PV,21088,432
46,SQ14,24429,586
58,Zone L1 (Station 1),33958,10099
59,Zone L1(Station 2),25929,2074


In [7]:
inv[inv['start'] >= '2023-01-01'][['station', 'has_optimizer', 'start']]

,station,has_optimizer,start
9,SQ Block P,True,2023-07-01
17,Shaw Auditorium,True,2023-08-01
24,UG Hall8,True,2023-03-02
25,UG Hall9,True,2023-02-11


In [8]:
inv[inv['end'] >= '2024-01-01'][['station', 'end']]

,station,end
58,Zone L1 (Station 1),2024-01-01 03:00:00


### Save the table

In [9]:
inv.to_csv('../results/tables/01_station_inventory.csv', index=False)